In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torchvision.models import vit_b_16
from torch.utils.data import DataLoader
import copy
import random
import numpy as np

# =====================================================
# Setup
# =====================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# =====================================================
# Data
# =====================================================
def get_cifar10(batch_size=32):

    transform_train = transforms.Compose([
        transforms.Resize(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5,)*3, (0.5,)*3)
    ])

    transform_test = transforms.Compose([
        transforms.Resize(224),
        transforms.ToTensor(),
        transforms.Normalize((0.5,)*3, (0.5,)*3)
    ])

    trainset = torchvision.datasets.CIFAR10(
        root="./data", train=True, download=True, transform=transform_train
    )

    testset = torchvision.datasets.CIFAR10(
        root="./data", train=False, download=True, transform=transform_test
    )

    trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
    testloader = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

    return trainloader, testloader

train_dl, test_dl = get_cifar10()

# =====================================================
# Evaluation
# =====================================================
@torch.no_grad()
def evaluate(model):
    model.eval()
    correct = 0
    total = 0
    for x, y in test_dl:
        x, y = x.to(device), y.to(device)
        out = model(x)
        pred = out.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return correct / total

# =====================================================
# Training
# =====================================================
def train(model, epochs=20):
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)
    scaler = torch.amp.GradScaler("cuda")

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()

            with torch.amp.autocast("cuda"):
                out = model(x)
                loss = F.cross_entropy(out, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_dl):.4f}")

# =====================================================
# Sparse Wrapper
# =====================================================
class SparseViT(nn.Module):
    def __init__(self, base_model, keep_ratio=0.5):
        super().__init__()
        self.base = base_model
        self.keep_ratio = keep_ratio

    def forward(self, x):
        x = self.base._process_input(x)
        B, N, C = x.shape

        cls_token = self.base.class_token.expand(B, -1, -1)
        x = torch.cat((cls_token, x), dim=1)
        x = x + self.base.encoder.pos_embedding

        scores = x.norm(dim=-1)
        k = int(N * self.keep_ratio)
        topk = torch.topk(scores[:, 1:], k, dim=1).indices + 1

        cls_idx = torch.zeros(B, 1, dtype=torch.long, device=x.device)
        keep_idx = torch.cat([cls_idx, topk], dim=1)

        x = torch.gather(x, 1, keep_idx.unsqueeze(-1).expand(-1, -1, C))
        x = self.base.encoder.layers(x)
        x = self.base.encoder.ln(x)

        cls = x[:, 0]
        return self.base.heads(cls)

# =====================================================
# 1️⃣ Train Dense
# =====================================================
print("\nTraining Dense ViT...")
dense = vit_b_16(weights="IMAGENET1K_V1")
dense.heads.head = nn.Linear(dense.heads.head.in_features, 10)

train(dense, epochs=20)
dense_acc = evaluate(dense)
print("Dense Accuracy:", dense_acc)

torch.save(dense.state_dict(), "dense_vit.pt")
print("Dense model saved.")

# =====================================================
# 2️⃣ Train Sparse
# =====================================================
print("\nTraining Sparse ViT...")
sparse = SparseViT(copy.deepcopy(dense), keep_ratio=0.5)

train(sparse, epochs=20)
sparse_acc = evaluate(sparse)
print("Sparse Accuracy:", sparse_acc)

torch.save(sparse.state_dict(), "sparse_vit.pt")
print("Sparse model saved.")

print("\n===== RESULTS =====")
print("Dense:", dense_acc)
print("Sparse:", sparse_acc)

Using device: cuda


100%|██████████| 170M/170M [00:03<00:00, 48.8MB/s] 



Training Dense ViT...
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 222MB/s] 


Epoch 1/20 - Loss: 0.1673
Epoch 2/20 - Loss: 0.0610
Epoch 3/20 - Loss: 0.0456
Epoch 4/20 - Loss: 0.0325
Epoch 5/20 - Loss: 0.0254
Epoch 6/20 - Loss: 0.0240
Epoch 7/20 - Loss: 0.0227
Epoch 8/20 - Loss: 0.0184
Epoch 9/20 - Loss: 0.0182
Epoch 10/20 - Loss: 0.0181
Epoch 11/20 - Loss: 0.0169
Epoch 12/20 - Loss: 0.0144
Epoch 13/20 - Loss: 0.0136
Epoch 14/20 - Loss: 0.0161
Epoch 15/20 - Loss: 0.0128
Epoch 16/20 - Loss: 0.0120
Epoch 17/20 - Loss: 0.0126
Epoch 18/20 - Loss: 0.0114
Epoch 19/20 - Loss: 0.0113
Epoch 20/20 - Loss: 0.0113
Dense Accuracy: 0.9689
Dense model saved.

Training Sparse ViT...
Epoch 1/20 - Loss: 0.1910
Epoch 2/20 - Loss: 0.0964
Epoch 3/20 - Loss: 0.0692
Epoch 4/20 - Loss: 0.0513
Epoch 5/20 - Loss: 0.0441
Epoch 6/20 - Loss: 0.0371
Epoch 7/20 - Loss: 0.0367
Epoch 8/20 - Loss: 0.0328
Epoch 9/20 - Loss: 0.0293
Epoch 10/20 - Loss: 0.0307
Epoch 11/20 - Loss: 0.0255
Epoch 12/20 - Loss: 0.0221
Epoch 13/20 - Loss: 0.0247
Epoch 14/20 - Loss: 0.0202
Epoch 15/20 - Loss: 0.0238
Epoch 1